In [25]:
spark

In [ ]:
import os
import sys

# Ensure our custom modules can be imported
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from src.utils.kafka_io import read_kafka_stream
from src.utils.stream_io import write_stream_table
from src.streaming.landing import transform_landing

In [27]:
spark.sql("CREATE DATABASE IF NOT EXISTS landing")

DataFrame[]

In [28]:


spark.sql("""
    CREATE TABLE IF NOT EXISTS landing.raw_events (
        key BINARY,
        value STRING,
        topic STRING,
        partition INT,
        offset BIGINT,
        kafka_timestamp TIMESTAMP,
        landing_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/landing/raw_events'
""")


DataFrame[]

In [ ]:


# 1. Fetch secrets securely from the Docker environment
BOOTSTRAP_SERVERS = os.environ.get("KAFKA_BOOTSTRAP_SERVERS")
API_KEY = os.environ.get("KAFKA_API_KEY")
API_SECRET = os.environ.get("KAFKA_API_SECRET")

# Catch missing variables so we don't accidentally pass 'None'
if not all([BOOTSTRAP_SERVERS, API_KEY, API_SECRET]):
    raise ValueError("Missing Kafka credentials! Check your .env setup.")

# 2. Read from Kafka
raw_kafka_df = read_kafka_stream(
    spark=spark,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    topic="orders",
    username=API_KEY,
    password=API_SECRET
)

# 3. Transform
landing_df = transform_landing(raw_kafka_df)

# 4. Write
landing_query = write_stream_table(
    df=landing_df,
    query_name="landing_raw_events_stream",
    checkpoint_path="/opt/spark-data/checkpoints/landing/v1/raw_events",
    table_name="landing.raw_events",
    trigger_time="10 seconds"
)


In [34]:
spark.sql("select * from landing.raw_events").show(truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [35]:
for query in spark.streams.active:
    print(query.name)

landing_raw_events_stream
